# AI Research Agent

연구 주제 하나를 입력하면 에이전트가 **주제 이해 → 조사 계획 → 자료 탐색(Tool Use) → 자료 평가 → 비교·종합(RAG) → 자기 비판(Reflection) → 재계획(Replanning) → Research Gap·향후 연구 방향 제안 → 인용 검증·수정**까지 스스로 수행

**사용법 (Colab)**
1. 아래 ① 셀에 연구 주제를 입력 (API 키는 비워두면 Colab 왼쪽 🔑 *보안 비밀*의 `ANTHROPIC_API_KEY` → 환경변수 → 직접 입력 순으로 찾습니다.)
2. 메뉴 **런타임 → 모두 실행** (Ctrl+F9)
3. 맨 아래 셀에 조사 결과 보고서가 출력되고, `/content/research_output/<실행시각_주제>/` 폴더에 보고서(.md·.html)·근거표(.csv)·수행기록(.json)과 묶음 파일(.zip) 저장

**실행 전 확인**
- **Anthropic API 키(유료)가 필요합니다.** 검색 도구 5종은 키 없이 동작합니다.
- 실행할 때마다 새 폴더에 저장되므로 다른 주제의 결과와 섞이지 않습니다.

In [ ]:
# ① 연구 주제 입력
RESEARCH_TOPIC = "생성형 AI 활용이 대학원생의 연구 생산성과 연구 품질에 미치는 영향"

ANTHROPIC_API_KEY = ""   # 비워두면: Colab 보안 비밀(🔑 ANTHROPIC_API_KEY) → 환경변수 → 실행 시 직접 입력

# ── 선택 설정 ──
MODEL = "claude-opus-5-5"          # 사용할 Claude 모델
MAX_REFLECTION_ROUNDS = 2          # 자기 비판 → 재계획 → 추가 조사 반복 최대 횟수
MAX_TOOL_STEPS = 4                 # 하위 질문 하나당 검색 도구 호출 라운드 최대 횟수
RESULTS_PER_SEARCH = 6             # 검색 1회당 가져올 문헌 수
YEAR_FROM = None                   # None: 에이전트가 주제를 보고 검색 시작 연도 결정 / 0: 제한 없음 / 2020: 2020년 이후만
OUTPUT_LANGUAGE = "ko"             # "auto"면 주제 언어로 보고서 작성 ("ko", "en" 등 지정 가능)
HUMAN_IN_THE_LOOP = False          # True면 조사 계획을 사람이 검토·수정한 뒤 진행
VERIFY_SAMPLE = 10                 # 인용 검증할 문장 수 (보고서 전체에서 고르게 추출, 미지지 문장은 수정 후 재검증)
OPENALEX_EMAIL = ""                # (선택) OpenAlex polite pool용 이메일 — 비워둬도 동작

In [ ]:
# ② 패키지 설치 (Colab 기준, 1회)
%pip install -q anthropic ddgs scikit-learn pandas requests markdown-it-py

## 설계 개요

```
 [연구 주제]
     │
 ① Understand ── 핵심 개념·변수·모집단·모호성 파악, 영어 검색어 생성, 검색 기간 결정
     │
 ② Plan ──────── 4~6개 하위 질문(관점이 서로 다르게 + 반대 증거 관점 필수)
     │            └ (옵션) Human-in-the-loop 계획 검토
     ▼
 ③ Act (Tool Use 루프, 하위 질문별)
     │   Claude가 스스로 도구·검색어를 고르고, 결과를 보고 다음 검색을 결정
     │   한 번에 요청한 여러 검색은 서로 다른 DB끼리 병렬 실행, 반복 실패한 DB는 자동 제외
     │   OpenAlex · Semantic Scholar · arXiv · Europe PMC · Web(DuckDuckGo)
     ▼
 ④ Evaluate ──── 관련성(LLM) + 신뢰도(동료심사·피인용·최신성, 코드) + 연구설계 가중치
     │            → 연구유형·방법·표본·핵심결과·결과방향 구조화 추출 (Evidence Store, 판본 중복 병합)
     ▼
 ⑤ Synthesize (RAG) ─ 하위 질문별 TF-IDF 검색으로 근거만 골라 [S#] 인용 종합
     ▼
 ⑥ Reflect (Critic) ─ 커버리지·편향·근거강도 비판 → 부족하면
     │                 추가 검색어/새 하위 질문 생성(Replanning) → ③으로 복귀
     ▼
 ⑦ Report ────── 종합에 쓴 근거 전부 + 상위 근거로 비교·종합, Research Gap, 향후 연구 설계 제안
     ▼
 ⑧ Verify ────── 없는 인용 제거 → 인용 문장 ↔ 초록 전문 대조 → 미지지 문장 수정 → 재검증
```

In [ ]:
# ③ 공통 설정·로깅·LLM 래퍼
import os, re, json, math, time, random, textwrap, threading, unicodedata, shutil
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field, asdict, fields
from datetime import datetime

import requests
import anthropic

CURRENT_YEAR = datetime.now().year
# 맨 위 셀에서 선택 설정 줄을 지웠거나 셀 실행 순서가 꼬여도 동작하도록 기본값 보충
for _k, _v in dict(MODEL="claude-opus-5-5", MAX_REFLECTION_ROUNDS=2, MAX_TOOL_STEPS=4, RESULTS_PER_SEARCH=6,
                   YEAR_FROM=None, OUTPUT_LANGUAGE="auto", HUMAN_IN_THE_LOOP=False, VERIFY_SAMPLE=10,
                   OPENALEX_EMAIL="").items():
    globals().setdefault(_k, _v)
OUTPUT_DIR = "/content/research_output" if os.path.isdir("/content") else "research_output"
TRACE = []  # 에이전트 수행 기록 (설계 평가·디버깅용)


def log(stage, msg, data=None):
    TRACE.append({"time": datetime.now().isoformat(timespec="seconds"), "stage": stage, "msg": msg, "data": data})
    print(f"[{stage:<10}] {msg}")


def resolve_api_key():
    if ANTHROPIC_API_KEY.strip():
        return ANTHROPIC_API_KEY.strip()
    try:
        from google.colab import userdata
        key = userdata.get("ANTHROPIC_API_KEY")
        if key:
            return key
    except Exception:
        pass
    if os.environ.get("ANTHROPIC_API_KEY"):
        return os.environ["ANTHROPIC_API_KEY"]
    import getpass
    return getpass.getpass("Anthropic API Key를 입력하세요: ").strip()


# JSON Schema 축약 헬퍼 (구조화 출력용)
S = {"type": "string"}
I = {"type": "integer"}
B = {"type": "boolean"}
def O(**props):
    return {"type": "object", "properties": props, "required": list(props), "additionalProperties": False}
def A(item):
    return {"type": "array", "items": item}
def E(*values):
    return {"type": "string", "enum": list(values)}


class LLM:
    """Claude 호출 래퍼: 구조화 출력, 사용량 집계, 거절(refusal) 시 서버측 fallback."""

    def __init__(self, api_key, model):
        self.client = anthropic.Anthropic(api_key=api_key, max_retries=4)
        self.model = model
        self.use_fallback = True
        self.usage = {"calls": 0, "input_tokens": 0, "output_tokens": 0}

    def _create(self, stream=False, **kw):
        kw.setdefault("model", self.model)
        if self.use_fallback:
            try:
                extra = dict(betas=["server-side-fallback-2026-07-01"], fallbacks="default")
                if stream:
                    with self.client.beta.messages.stream(**kw, **extra) as s:
                        resp = s.get_final_message()
                else:
                    resp = self.client.beta.messages.create(**kw, **extra)
                return self._track(resp)
            except (anthropic.BadRequestError, TypeError) as e:
                if "fallback" not in str(e).lower():
                    raise
                self.use_fallback = False  # 이 계정/환경에서 미지원 → 일반 호출로 전환
        if stream:
            with self.client.messages.stream(**kw) as s:
                return self._track(s.get_final_message())
        return self._track(self.client.messages.create(**kw))

    def _track(self, resp):
        self.usage["calls"] += 1
        self.usage["input_tokens"] += resp.usage.input_tokens
        self.usage["output_tokens"] += resp.usage.output_tokens
        if resp.stop_reason == "refusal":
            log("warn", "모델이 이 요청을 거절했습니다(refusal).")
        return resp

    @staticmethod
    def _text(resp):
        return "".join(b.text for b in resp.content if b.type == "text").strip()

    def text(self, system, prompt, effort="medium", max_tokens=16000, stream=False):
        resp = self._create(stream=stream, system=system, max_tokens=max_tokens,
                            output_config={"effort": effort},
                            messages=[{"role": "user", "content": prompt}])
        return self._text(resp)

    def json(self, system, prompt, schema, effort="medium", max_tokens=16000):
        resp = self._create(system=system, max_tokens=max_tokens,
                            output_config={"effort": effort, "format": {"type": "json_schema", "schema": schema}},
                            messages=[{"role": "user", "content": prompt}])
        if resp.stop_reason in ("refusal", "max_tokens"):
            log("warn", f"구조화 출력 실패(stop_reason={resp.stop_reason})")
            return None
        try:
            return json.loads(self._text(resp))
        except json.JSONDecodeError:
            return None

    def tool_turn(self, system, messages, tools, effort="low", max_tokens=8000):
        return self._create(system=system, messages=messages, tools=tools, max_tokens=max_tokens,
                            output_config={"effort": effort})

In [ ]:
# ④ 도구(Tools): 학술 DB·웹 검색  — 모두 API 키 불필요
HTTP = requests.Session()
HTTP.headers["User-Agent"] = "EduResearchAgent/1.0 (AI agent class project)"


def http_get(url, params, as_json=True, retries=3, timeout=25):
    last = None
    for i in range(retries):
        try:
            r = HTTP.get(url, params=params, timeout=timeout)
            if r.status_code == 429:  # 속도 제한 → 지수 백오프
                time.sleep(2 ** (i + 1) + random.random())
                last = RuntimeError("429 rate limited")
                continue
            r.raise_for_status()
            return r.json() if as_json else r.text
        except Exception as e:
            last = e
            time.sleep(1 + i)
    raise RuntimeError(f"요청 실패: {last}")


@dataclass
class Source:
    title: str
    authors: list
    year: int | None
    venue: str
    abstract: str
    url: str
    doi: str = ""
    citations: int | None = None
    origin: str = ""          # 어느 도구에서 왔는지
    kind: str = "academic"    # peer-reviewed | academic | preprint | web
    sid: str = ""
    queries: set = field(default_factory=set)
    subqs: set = field(default_factory=set)
    evaluated: bool = False
    relevance: int = 0        # 0~3 (LLM)
    credibility: float = 0.0  # 0~1 (규칙)
    score: float = 0.0        # 종합 점수
    study_type: str = ""
    method: str = ""
    sample: str = ""
    key_finding: str = ""
    direction: str = ""
    limitation: str = ""

    def short_cite(self):
        names = [n for n in self.authors if n and n.split()]
        parts = names[0].replace(",", " ").split() if names else []
        # "John Smith" → Smith, "Kim J"(Europe PMC 형식) → Kim
        a = (parts[-1] if len(parts[-1].strip(".")) > 2 or len(parts) == 1 else parts[0]) if parts else (self.venue or "n.a.")
        etal = " et al." if len(names) > 1 else ""
        return f"{a}{etal} ({self.year or 'n.d.'})"


KIND_RANK = {"web": 0, "preprint": 1, "academic": 2, "peer-reviewed": 3}
_SITE_SUFFIX = re.compile(r"\s+[-|–—]\s+[^-|–—]{2,40}$")   # 웹 제목 끝의 "- SSRN", "| ScienceDirect" 등
_ELLIPSIS = re.compile(r"\s*(\.\.\.|…)\s*$")


def _norm_title(t, web=False):
    """제목 정규화 → (키, 잘림 여부). 합자(ﬀ)·사이트명 꼬리·말줄임표 차이로 같은 문헌이 따로 저장되는 것을 막음."""
    t = unicodedata.normalize("NFKC", t or "").strip()
    if web:
        t = _SITE_SUFFIX.sub("", t)
    truncated = bool(_ELLIPSIS.search(t))
    return re.sub(r"[^a-z0-9가-힣]", "", _ELLIPSIS.sub("", t).lower())[:120], truncated


class EvidenceStore:
    """수집 문헌 저장소: DOI/제목 기준 중복(같은 문헌의 여러 판본) 병합, S1, S2 ... ID 부여."""

    def __init__(self):
        self.items: dict[str, Source] = {}
        self._dois = {}
        self._titles = {}   # 제목 키 → (sid, 잘린 제목 여부)
        self.merged = 0     # 기존 문헌과 같은 것으로 판정되어 병합된 검색 결과 수

    def _match(self, doi, tkey, truncated):
        if doi and doi in self._dois:
            return self._dois[doi]
        if tkey in self._titles:
            return self._titles[tkey][0]
        if len(tkey) >= 35:  # "제목 앞부분 ..." 처럼 잘린 웹 제목은 앞부분이 같으면 같은 문헌
            for k, (sid, trunc) in self._titles.items():
                if (truncated and k.startswith(tkey)) or (trunc and tkey.startswith(k)):
                    return sid
        return None

    def add(self, src: Source, query="", subq=""):
        if not src.title:
            return None
        src.authors = [a.strip() for a in (src.authors or []) if a and a.strip()]
        doi = (src.doi or "").lower().replace("https://doi.org/", "")
        tkey, truncated = _norm_title(src.title, web=src.kind == "web")
        existing = self._match(doi, tkey, truncated)
        if existing:
            s = self.items[existing]
            self.merged += 1
            if not s.evaluated and KIND_RANK.get(src.kind, 0) > KIND_RANK.get(s.kind, 0):
                # 웹 사본보다 학술 판본의 서지 정보를 우선 사용
                s.title, s.venue, s.url, s.kind, s.origin = src.title, src.venue, src.url, src.kind, src.origin
                s.authors, s.year = src.authors or s.authors, src.year or s.year
            if len(src.abstract or "") > len(s.abstract or ""):
                s.abstract = src.abstract
            s.citations = max(s.citations or 0, src.citations or 0) or s.citations
            s.doi = s.doi or src.doi
        else:
            src.sid = f"S{len(self.items) + 1}"
            self.items[src.sid] = s = src
        if doi:
            self._dois.setdefault(doi, s.sid)
        if tkey:
            self._titles.setdefault(tkey, (s.sid, truncated))
        if query:
            s.queries.add(query)
        if subq:
            s.subqs.add(subq)
        return s

    def __len__(self):
        return len(self.items)

    def included(self):
        return sorted([s for s in self.items.values() if s.evaluated and s.relevance >= 2],
                      key=lambda s: -s.score)


def _openalex_abstract(inv):
    if not inv:
        return ""
    pos = sorted((p, w) for w, ps in inv.items() for p in ps)
    return " ".join(w for _, w in pos)


def search_openalex(query, limit=RESULTS_PER_SEARCH, year_from=None):
    params = {"search": query, "per-page": limit,
              "select": "doi,title,publication_year,primary_location,authorships,cited_by_count,abstract_inverted_index,type,id"}
    if year_from:
        params["filter"] = f"from_publication_date:{year_from}-01-01"
    if OPENALEX_EMAIL:
        params["mailto"] = OPENALEX_EMAIL
    out = []
    for w in http_get("https://api.openalex.org/works", params).get("results", []):
        src = (w.get("primary_location") or {}).get("source") or {}
        stype = src.get("type") or ""
        kind = "preprint" if (w.get("type") == "preprint" or stype == "repository") else \
               "peer-reviewed" if stype in ("journal", "conference") else "academic"
        out.append(Source(
            title=w.get("title") or "", year=w.get("publication_year"), venue=src.get("display_name") or "",
            authors=[a["author"]["display_name"] for a in (w.get("authorships") or [])[:6] if a.get("author")],
            abstract=_openalex_abstract(w.get("abstract_inverted_index")), doi=w.get("doi") or "",
            url=w.get("doi") or w.get("id") or "", citations=w.get("cited_by_count"), origin="OpenAlex", kind=kind))
    return out


def search_semantic_scholar(query, limit=RESULTS_PER_SEARCH, year_from=None):
    params = {"query": query, "limit": limit,
              "fields": "title,abstract,year,venue,authors,citationCount,externalIds,url,publicationTypes"}
    if year_from:
        params["year"] = f"{year_from}-"
    out = []
    for p in http_get("https://api.semanticscholar.org/graph/v1/paper/search", params).get("data", []) or []:
        types = p.get("publicationTypes") or []
        venue = p.get("venue") or ""
        kind = "preprint" if "arxiv" in venue.lower() else \
               "peer-reviewed" if ("JournalArticle" in types or "Conference" in types) else "academic"
        doi = (p.get("externalIds") or {}).get("DOI") or ""
        out.append(Source(
            title=p.get("title") or "", year=p.get("year"), venue=venue,
            authors=[a.get("name", "") for a in (p.get("authors") or [])[:6]],
            abstract=p.get("abstract") or "", doi=f"https://doi.org/{doi}" if doi else "",
            url=p.get("url") or "", citations=p.get("citationCount"), origin="SemanticScholar", kind=kind))
    return out


def search_arxiv(query, limit=RESULTS_PER_SEARCH, year_from=None):
    import xml.etree.ElementTree as ET
    ns = {"a": "http://www.w3.org/2005/Atom"}
    time.sleep(3)  # arXiv API 권장: 요청 간 3초 간격 (안 지키면 429)
    xml = http_get("https://export.arxiv.org/api/query",
                   {"search_query": f"all:{query}", "max_results": limit, "sortBy": "relevance"},
                   as_json=False, retries=2, timeout=60)
    out = []
    for e in ET.fromstring(xml).findall("a:entry", ns):
        year = int((e.findtext("a:published", "", ns) or "0")[:4] or 0) or None
        if year_from and year and year < year_from:
            continue
        out.append(Source(
            title=" ".join((e.findtext("a:title", "", ns) or "").split()), year=year, venue="arXiv",
            authors=[a.findtext("a:name", "", ns) for a in e.findall("a:author", ns)][:6],
            abstract=" ".join((e.findtext("a:summary", "", ns) or "").split()),
            url=e.findtext("a:id", "", ns), origin="arXiv", kind="preprint"))
    return out


def search_europe_pmc(query, limit=RESULTS_PER_SEARCH, year_from=None):
    q = f"({query})" + (f" AND PUB_YEAR:[{year_from} TO {CURRENT_YEAR}]" if year_from else "")
    data = http_get("https://www.ebi.ac.uk/europepmc/webservices/rest/search",
                    {"query": q, "format": "json", "resultType": "core", "pageSize": limit})
    out = []
    for r in data.get("resultList", {}).get("result", []):
        doi = r.get("doi") or ""
        out.append(Source(
            title=r.get("title") or "", year=int(r["pubYear"]) if r.get("pubYear") else None,
            venue=((r.get("journalInfo") or {}).get("journal") or {}).get("title", "") or r.get("source", ""),
            authors=[a.strip() for a in (r.get("authorString") or "").split(",")][:6],
            abstract=re.sub(r"<[^>]+>", "", r.get("abstractText") or ""),
            doi=f"https://doi.org/{doi}" if doi else "",
            url=f"https://doi.org/{doi}" if doi else f"https://europepmc.org/article/{r.get('source')}/{r.get('id')}",
            citations=r.get("citedByCount"), origin="EuropePMC",
            kind="preprint" if r.get("source") == "PPR" else "peer-reviewed"))
    return out


def search_web(query, limit=RESULTS_PER_SEARCH, year_from=None):
    # 웹 결과는 출판 연도를 알 수 없어 기간 필터 대신 평가 단계의 포함 기준(검색 기간 포함)으로 거름
    try:
        from ddgs import DDGS
    except ImportError:
        from duckduckgo_search import DDGS
    try:
        hits = DDGS().text(query, max_results=limit) or []
    except Exception as e:
        if "no results" in str(e).lower():  # 결과 0건은 오류가 아니라 빈 결과
            return []
        raise
    out = []
    for r in hits:
        out.append(Source(title=r.get("title", ""), authors=[], year=None, venue=re.sub(r"^https?://([^/]+).*", r"\1", r.get("href", "")),
                          abstract=r.get("body", ""), url=r.get("href", ""), origin="Web", kind="web"))
    return out


TOOL_FUNCS = {
    "search_openalex": search_openalex,
    "search_semantic_scholar": search_semantic_scholar,
    "search_arxiv": search_arxiv,
    "search_europe_pmc": search_europe_pmc,
    "search_web": search_web,
}
_TOOL_DESC = {
    "search_openalex": "OpenAlex: 2억+ 학술문헌, 전 분야(사회과학·교육·경영·공학 등). 기본 검색 도구로 가장 먼저 사용. 영어 키워드 권장.",
    "search_semantic_scholar": "Semantic Scholar: 전 분야 학술문헌, 특히 CS/AI에 강함, 피인용 수 제공. 영어 키워드.",
    "search_arxiv": "arXiv: 최신 프리프린트(CS, AI, 물리, 수학, 통계, 경제). 동료심사 전 최신 동향 확인용.",
    "search_europe_pmc": "Europe PMC: 의학·생명과학·보건·심리 문헌(PubMed 포함). 건강·의료·임상 관련 질문에 사용.",
    "search_web": "웹 검색(DuckDuckGo): 정책보고서, 기관 통계, 뉴스, 회색문헌, 한국어 자료. 학술 DB에 없는 맥락·최신 사례 확인용.",
}
TOOL_LOCKS = {n: threading.Lock() for n in TOOL_FUNCS}  # 같은 DB는 순차 호출(속도 제한 준수), 다른 DB끼리는 병렬
TOOLS = [{"name": n, "description": d, "strict": True,
          "input_schema": O(query={"type": "string", "description": "검색어 (학술 DB는 영어 키워드 3~8단어 권장)"})}
         for n, d in _TOOL_DESC.items()]

In [ ]:
# ⑤ Research Agent
SYS_BASE = ("You are a rigorous research assistant agent helping a graduate student. "
            "Be precise, evidence-based and critical. Never invent sources, numbers or findings.")
# 이 에이전트는 초록만 읽으므로, 초록에 없는 정보를 "논문이 보고하지 않았다"고 단정하지 않게 하는 공통 규칙
ABSTRACT_RULE = ("You only see titles/abstracts. If a detail (sample size, measure, effect size) is not in the abstract, "
                 "say it is 'not stated in the abstract' - never claim the study itself did not report it.")

STUDY_TYPES = ["meta-analysis/systematic review", "RCT/experiment", "quasi-experiment", "longitudinal",
               "survey/correlational", "qualitative", "mixed methods", "case study",
               "theoretical/conceptual", "review/commentary", "report/web article", "other"]
DESIGN_WEIGHT = {"meta-analysis/systematic review": 1.0, "RCT/experiment": 0.9, "quasi-experiment": 0.75,
                 "longitudinal": 0.7, "mixed methods": 0.6, "survey/correlational": 0.55, "qualitative": 0.5,
                 "case study": 0.4, "review/commentary": 0.4, "theoretical/conceptual": 0.35,
                 "report/web article": 0.25, "other": 0.3}
KIND_WEIGHT = {"peer-reviewed": 0.6, "academic": 0.45, "preprint": 0.35, "web": 0.15}
TOOL_FAIL_LIMIT = 3          # 같은 도구가 연속 3회 실패하면 이번 실행에서 제외 (예: Semantic Scholar 429 반복)
MAX_REPORT_EVIDENCE = 100    # 최종 보고서에 넘길 근거 상한 (종합에 쓴 근거는 상한과 무관하게 전부 포함)
VERDICTS = ["supported", "partially supported", "not in source text", "contradicted"]

_CITE = re.compile(r"\[S\d+(?:\s*,\s*S\d+)*\]")
_SENT_END = re.compile(r"(?<=[.!?。])(?<!\bal\.)(?<!e\.g\.)(?<!i\.e\.)(?<!vs\.)\s+")


class ResearchAgent:
    def __init__(self, topic, llm: LLM):
        self.topic = topic.strip()
        self.llm = llm
        self.store = EvidenceStore()
        self.understanding = {}
        self.plan = {}
        self.syntheses = {}
        self.reflections = []
        self.lang = "ko"
        self.year_from = None
        self.tool_failures = {n: 0 for n in TOOL_FUNCS}
        self.disabled_tools = set()
        self.report_evidence = []

    # ── ① 주제 이해 ──
    def understand(self):
        schema = O(language=S, topic_en=S, discipline=S, research_type=S,
                   key_concepts=A(O(term=S, english=S, synonyms=A(S))),
                   independent_variables=A(S), dependent_variables=A(S), moderators_or_context=A(S),
                   population=S, ambiguities=A(S), scope_decision=S, search_start_year=I, search_window_reason=S)
        u = self.llm.json(SYS_BASE, f"""Analyze this research topic before any searching.
Topic: "{self.topic}"

- language: ISO code of the topic text (e.g. "ko", "en").
- topic_en: faithful English translation.
- discipline / research_type (e.g. causal-effect, descriptive, comparative, methodological, design).
- key_concepts with English academic terms and synonyms useful as search keywords.
- variables (independent / dependent / moderators-context), target population.
- ambiguities: terms that can be interpreted several ways, and scope_decision: how you will scope them
  (e.g. operational definitions such as how 'quality' could be measured).
- search_start_year: earliest publication year worth searching for THIS topic (e.g. when the technology or
  phenomenon appeared, so that early foundational studies are kept). Use 0 if older foundational work matters
  and no year limit should be applied. Current year: {CURRENT_YEAR}. search_window_reason: one sentence.""",
                          schema, effort="medium")
        if not u:
            raise RuntimeError("주제 이해 단계 실패")
        self.understanding = u
        self.lang = u["language"] if OUTPUT_LANGUAGE == "auto" else OUTPUT_LANGUAGE
        # 검색 기간은 여기 한 곳에서 정하고 모든 학술 DB 검색·포함 기준에 같은 값을 씀
        y = YEAR_FROM if YEAR_FROM is not None else u["search_start_year"]
        self.year_from = y if y and 1900 < y <= CURRENT_YEAR else None
        src = "사용자 설정" if YEAR_FROM is not None else f"에이전트 판단: {u['search_window_reason'][:120]}"
        log("Understand", f"분야={u['discipline']} | 유형={u['research_type']} | 개념={', '.join(c['english'] for c in u['key_concepts'])}", u)
        log("Understand", f"검색 기간: {f'{self.year_from}년 이후' if self.year_from else '제한 없음'} ({src})")

    # ── ② 계획 수립 ──
    def make_plan(self, feedback=""):
        schema = O(sub_questions=A(O(id=S, question=S, perspective=S, rationale=S,
                                     search_queries=A(S), preferred_tools=A(E(*TOOL_FUNCS)))),
                   inclusion_criteria=A(S), success_criteria=A(S))
        window = f"publications from {self.year_from} onward" if self.year_from else "no publication-year limit"
        p = self.llm.json(SYS_BASE, f"""Create an investigation plan for the topic.
Topic analysis: {json.dumps(self.understanding, ensure_ascii=False)}
Search window (already applied to all academic databases): {window}
{f'Human reviewer feedback to incorporate: {feedback}' if feedback else ''}

Rules:
- 4 to 6 sub-questions with ids Q1, Q2, ... Each must cover a DIFFERENT perspective, chosen to fit THIS topic, e.g.:
  definitions/measurement, current state/adoption, empirical effects, mechanisms, moderators/context,
  risks/limitations, methodology used in prior studies.
- At least one sub-question MUST look for counter-evidence, null/negative results or risks (avoid confirmation bias).
- 2-3 search_queries per sub-question: concise English academic keyword queries (3-8 words) using synonyms.
  Add a query in the topic's language only if local/national context matters.
- preferred_tools: pick tools appropriate for the discipline. Tools: {json.dumps(_TOOL_DESC, ensure_ascii=False)}
- inclusion_criteria: what counts as relevant evidence; any time criterion must match the search window above.
  success_criteria: when the investigation is "good enough".""",
                          schema, effort="medium")
        if not p:
            raise RuntimeError("계획 수립 단계 실패")
        self.plan = p
        log("Plan", f"하위 질문 {len(p['sub_questions'])}개 수립", p)
        for q in p["sub_questions"]:
            print(f"    {q['id']} [{q['perspective']}] {q['question']}")

    def human_review(self):
        if not HUMAN_IN_THE_LOOP:
            return
        fb = input("\n✋ 조사 계획을 검토하세요. 수정 의견을 입력하거나 Enter로 승인: ").strip()
        if fb:
            log("Human", f"사용자 피드백 반영하여 재계획: {fb}")
            self.make_plan(feedback=fb)

    # ── ③ 자료 탐색: Tool Use 루프 ──
    def _fetch(self, name, query):
        """도구 실행만 담당(스레드에서 병렬 실행). 같은 DB는 잠금으로 순차 호출해 속도 제한을 지킴."""
        if name in self.disabled_tools:
            return None, f"{name} is unavailable for the rest of this run after repeated failures"
        with TOOL_LOCKS[name]:
            try:
                return TOOL_FUNCS[name](query, year_from=self.year_from), None
            except Exception as e:
                return None, str(e)

    def _store_results(self, name, query, subq, results, err):
        if err is not None:
            log("Tool", f"{name}('{query}') 오류: {err}")
            self.tool_failures[name] += 1
            if self.tool_failures[name] >= TOOL_FAIL_LIMIT and name not in self.disabled_tools:
                self.disabled_tools.add(name)
                log("Tool", f"{name} 연속 {TOOL_FAIL_LIMIT}회 실패 → 이번 실행에서 제외")
            return f"ERROR: {err}. Try another tool or query.", True
        self.tool_failures[name] = 0
        lines = []
        for src in results:
            try:
                s = self.store.add(src, query=query, subq=subq)
            except Exception as e:
                log("Tool", f"결과 1건 건너뜀: {e}")
                continue
            if s:
                lines.append(f"[{s.sid}] {s.short_cite()} {s.title} | {s.venue} | cites={s.citations} | {s.kind}\n"
                             f"     {(s.abstract or 'no abstract')[:220]}")
        log("Tool", f"{subq} · {name}('{query}') → {len(results)}건 (누적 {len(self.store)})")
        return ("\n".join(lines) or "No results. Try broader or different keywords."), False

    def _run_tools(self, calls):
        """calls=[(도구, 검색어, 하위질문)]: 서로 다른 DB는 동시에 실행하고, 결과는 요청 순서대로 저장소에 넣음."""
        with ThreadPoolExecutor(max_workers=len(TOOL_FUNCS)) as ex:
            fetched = list(ex.map(lambda c: self._fetch(c[0], c[1]), calls))
        return [self._store_results(n, q, sq, *f) for (n, q, sq), f in zip(calls, fetched)]

    def investigate(self, sq):
        before = {s.sid for s in self.store.items.values() if sq["id"] in s.subqs}
        tools = [t for t in TOOLS if t["name"] not in self.disabled_tools]
        system = (SYS_BASE + " You are the SEARCH EXECUTOR. Use the search tools to collect evidence for ONE sub-question. "
                  "Look at each result list, then decide: refine keywords, switch database, or stop. "
                  "Aim for 6-12 clearly relevant, diverse sources (empirical studies, reviews, and counter-evidence). "
                  "When enough evidence is collected, stop calling tools and reply with a 2-3 sentence summary of what you found and what is missing.")
        msgs = [{"role": "user", "content":
                 f"Main topic: {self.understanding['topic_en']}\nSub-question {sq['id']}: {sq['question']}\n"
                 f"Perspective: {sq['perspective']}\nSuggested queries: {sq['search_queries']}\n"
                 f"Suggested tools: {sq['preferred_tools']}\nYou may run several searches at once (they run in parallel)."}]
        summary = ""
        for _ in range(MAX_TOOL_STEPS if tools else 0):
            resp = self.llm.tool_turn(system, msgs, tools)
            msgs.append({"role": "assistant", "content": resp.content})
            uses = [b for b in resp.content if b.type == "tool_use"]
            if resp.stop_reason != "tool_use" or not uses:
                summary = LLM._text(resp)
                break
            outs = self._run_tools([(u.name, u.input.get("query", ""), sq["id"]) for u in uses])
            msgs.append({"role": "user", "content": [{"type": "tool_result", "tool_use_id": u.id, "content": out, "is_error": err}
                                                     for u, (out, err) in zip(uses, outs)]})
        got = lambda: {s.sid for s in self.store.items.values() if sq["id"] in s.subqs} - before
        if len(got()) < 3:  # 안전장치: 도구 루프가 자료를 거의 못 찾으면 결정적 검색으로 보강
            log("Act", f"{sq['id']} 수집 부족({len(got())}건) → 기본 검색으로 보강")
            self._run_tools([("search_openalex", q, sq["id"]) for q in sq["search_queries"][:2]])
        log("Act", f"{sq['id']} 탐색 완료: 관련 후보 {len(got())}건. {summary[:150]}")

    # ── ④ 자료 평가 ──
    @staticmethod
    def _credibility(s: Source):
        c = KIND_WEIGHT.get(s.kind, 0.3)
        c += min(math.log10((s.citations or 0) + 1) / 3, 1) * 0.3   # 피인용 1000회 ≈ +0.3
        if s.year:
            c += 0.1 if s.year >= CURRENT_YEAR - 5 else 0.05 if s.year >= CURRENT_YEAR - 10 else 0
        return round(min(c, 1.0), 2)

    def evaluate(self, batch_size=8):
        todo = [s for s in self.store.items.values() if not s.evaluated]
        if not todo:
            return
        subqs = "\n".join(f"{q['id']}: {q['question']}" for q in self.plan["sub_questions"])
        schema = O(items=A(O(sid=S, relevance=I, relevant_subqs=A(S), study_type=E(*STUDY_TYPES),
                             method=S, sample=S, key_finding=S,
                             direction=E("positive", "negative", "mixed", "null", "not applicable"),
                             limitation=S)))
        for i in range(0, len(todo), batch_size):
            batch = todo[i:i + batch_size]
            docs = "\n\n".join(f"<{s.sid}> {s.title} ({s.year}, {s.venue}, {s.kind})\n{(s.abstract or 'NO ABSTRACT')[:1200]}"
                               for s in batch)
            res = self.llm.json(SYS_BASE, f"""Evaluate each source for the research topic using ONLY its title/abstract.
{ABSTRACT_RULE}
Topic: {self.understanding['topic_en']}
Sub-questions:
{subqs}
Inclusion criteria: {self.plan['inclusion_criteria']}

For every source return:
- relevance: 0=unrelated, 1=tangential, 2=relevant, 3=directly answers a sub-question
- relevant_subqs: ids it informs
- study_type, method (design/analysis, short), sample (who/how many; if absent write that it is not stated in the abstract)
- key_finding: one sentence with the main result as stated (include numbers/effect sizes only if present in the text;
  if the abstract gives no result, say so)
- direction: of the effect relative to the topic's main hypothesis
- limitation: main weakness visible from the abstract
Write method/sample/key_finding/limitation in {self.lang}.

Sources:
{docs}""", schema, effort="low")
            by_id = {r["sid"].strip("<>"): r for r in (res or {}).get("items", [])}
            for s in batch:
                r = by_id.get(s.sid)
                s.evaluated = True
                s.credibility = self._credibility(s)
                if not r:
                    continue
                s.relevance = max(0, min(3, r["relevance"]))
                s.subqs.update(q for q in r["relevant_subqs"] if q)
                s.study_type, s.method, s.sample = r["study_type"], r["method"], r["sample"]
                s.key_finding, s.direction, s.limitation = r["key_finding"], r["direction"], r["limitation"]
                s.score = round(0.55 * s.relevance / 3 + 0.30 * s.credibility
                                + 0.15 * DESIGN_WEIGHT.get(s.study_type, 0.3), 3)
        inc = self.store.included()
        log("Evaluate", f"평가 {len(todo)}건 → 채택(관련성≥2) 누적 {len(inc)}건 / 전체 {len(self.store)}건 "
                        f"(중복 판본 병합 누적 {self.store.merged}건)")

    # ── ⑤ 종합 (RAG) ──
    def retrieve(self, sq, k=10):
        """하위 질문별 근거 검색: TF-IDF 유사도 + 평가 점수 + 하위질문 태그 가중."""
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.metrics.pairwise import cosine_similarity
        pool = self.store.included()
        if not pool:
            return []
        docs = [f"{s.title} {s.abstract} {s.key_finding}" for s in pool]
        query = sq["question"] + " " + " ".join(sq["search_queries"])
        vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), sublinear_tf=True).fit(docs + [query])
        sims = cosine_similarity(vec.transform([query]), vec.transform(docs))[0]
        top = sims.max() or 1
        ranked = sorted(zip(pool, sims), key=lambda x: -(0.45 * x[1] / top + 0.35 * x[0].score
                                                          + (0.2 if sq["id"] in x[0].subqs else 0)))
        return [s for s, _ in ranked[:k]]

    def synthesize(self, sq):
        ev = self.retrieve(sq)
        if not ev:
            self.syntheses[sq["id"]] = {"answer": "근거 없음", "key_findings": [], "consensus": [], "disagreements": [],
                                        "evidence_strength": "insufficient", "gaps": ["관련 문헌을 찾지 못함"], "sources": []}
            return
        ctx = "\n\n".join(f"[{s.sid}] {s.short_cite()} | {s.kind} | {s.study_type} | {s.method} | sample: {s.sample} | "
                          f"direction: {s.direction} | credibility {s.credibility}\nTitle: {s.title}\n"
                          f"Finding: {s.key_finding}\nAbstract: {(s.abstract or '')[:700]}" for s in ev)
        schema = O(answer=S, key_findings=A(O(finding=S, sources=A(S))), consensus=A(S), disagreements=A(S),
                   evidence_strength=E("strong", "moderate", "weak", "insufficient"), gaps=A(S))
        out = self.llm.json(SYS_BASE, f"""Answer the sub-question using ONLY the evidence below. Cite every claim with source ids like [S3].
{ABSTRACT_RULE}
Sub-question {sq['id']}: {sq['question']}

Evidence:
{ctx}

- answer: 1 concise paragraph.
- key_findings: 3-6 items, each with the supporting source ids.
- consensus / disagreements: where studies agree or conflict, and WHY they might differ (design, sample, context, measurement).
- evidence_strength: judge by quantity, design quality and consistency. Several copies/versions of one study count as ONE study.
- gaps: what this evidence cannot answer.
Write in language "{self.lang}".""", schema, effort="medium")
        if out:
            out["sources"] = [s.sid for s in ev]
            self.syntheses[sq["id"]] = out
            log("Synthesize", f"{sq['id']} 종합 완료 (근거 {len(ev)}건, 강도={out['evidence_strength']})")

    # ── ⑥ 자기 비판 & 재계획 ──
    def _evidence_stats(self):
        inc = self.store.included()
        count = lambda key: {v: sum(1 for s in inc if getattr(s, key) == v) for v in {getattr(s, key) for s in inc}}
        years = [s.year for s in inc if s.year]
        return {"total_collected": len(self.store), "included": len(inc), "duplicates_merged": self.store.merged,
                "search_window_from": self.year_from, "unavailable_tools": sorted(self.disabled_tools),
                "by_kind": count("kind"), "by_study_type": count("study_type"), "by_direction": count("direction"),
                "year_range": [min(years), max(years)] if years else None,
                "per_subq": {q["id"]: sum(1 for s in inc if q["id"] in s.subqs) for q in self.plan["sub_questions"]}}

    def reflect(self, round_no, final=False):
        summary = {qid: {"answer": v["answer"][:600], "strength": v["evidence_strength"], "gaps": v["gaps"]}
                   for qid, v in self.syntheses.items()}
        history = [{"round": i + 1, "quality": r["quality_score"], "searched": [f["query"] for f in r["followup_queries"]],
                    "added_questions": [q["question"] for q in r["new_sub_questions"]]} for i, r in enumerate(self.reflections)]
        if final:
            budget = ("This is the FINAL check after the last search round: no more searching is possible. "
                      "Set should_continue=false, leave followup lists empty, and list the issues the final report must disclose.")
        else:
            left = MAX_REFLECTION_ROUNDS - round_no + 1
            budget = (f"Search rounds still available: {left} (including the one that runs right after this review if you continue). "
                      "Do not stop merely because of the round number; decide only on whether more searching would materially improve the result.")
        schema = O(quality_score=I, subq_assessment=A(O(id=S, coverage=E("good", "partial", "poor"), issue=S)),
                   bias_risks=A(S), should_continue=B, reason=S,
                   followup_queries=A(O(subq_id=S, query=S, tool=E(*TOOL_FUNCS))),
                   new_sub_questions=A(O(question=S, perspective=S, rationale=S, search_queries=A(S))))
        r = self.llm.json(SYS_BASE + " You are now the CRITIC reviewing the research so far.", f"""Critically review the investigation.
Topic: {self.understanding['topic_en']}
Plan: {json.dumps(self.plan['sub_questions'], ensure_ascii=False)}
Success criteria: {self.plan['success_criteria']}
Current syntheses: {json.dumps(summary, ensure_ascii=False)}
Evidence statistics: {json.dumps(self._evidence_stats(), ensure_ascii=False)}
Previous critic rounds (do not repeat the same searches): {json.dumps(history, ensure_ascii=False)}
{budget}

Check: coverage of each sub-question, one-sided evidence (e.g. only positive results), over-reliance on weak/old/web sources,
missing perspectives, missing study designs, duplicated versions of one study counted as several. quality_score 1-10.
should_continue=true ONLY if additional searching is likely to materially improve the result.
If continuing: up to 4 followup_queries (concise English keywords, choose the best tool; avoid unavailable_tools) and up to 2
new_sub_questions for important uncovered angles (leave lists empty otherwise).""", schema, effort="medium")
        if not r:
            return False
        if final:
            r.update(should_continue=False, followup_queries=[], new_sub_questions=[])
        self.reflections.append(r)
        label = "최종 점검" if final else f"라운드 {round_no}"
        log("Reflect", f"{label}: 품질 {r['quality_score']}/10, 계속={r['should_continue']} — {r['reason'][:150]}", r)
        for b in r["bias_risks"][:3]:
            print(f"    ⚠ {b}")
        return r["should_continue"]

    def replan_and_act(self, r):
        known = {q["id"] for q in self.plan["sub_questions"]}
        calls = [(fq["tool"], fq["query"], fq["subq_id"]) for fq in r["followup_queries"][:4] if fq["subq_id"] in known]
        self._run_tools(calls)
        affected = {sq for _, _, sq in calls}
        for nq in r["new_sub_questions"][:2]:
            sq = {"id": f"Q{len(self.plan['sub_questions']) + 1}", "preferred_tools": list(TOOL_FUNCS), **nq}
            self.plan["sub_questions"].append(sq)
            log("Replan", f"새 하위 질문 추가 {sq['id']}: {sq['question']}")
            self.investigate(sq)
            affected.add(sq["id"])
        self.evaluate()
        for sq in self.plan["sub_questions"]:
            if sq["id"] in affected:
                self.synthesize(sq)

    # ── ⑦ 최종 보고서 ──
    def _report_evidence(self):
        """보고서에 넘길 근거: 하위 질문 종합이 다룬 문헌은 전부 + 남는 자리는 점수순.
        (점수 상위만 넘기면 종합에서 쓴 문헌이 빠져 '근거 없음'으로 잘못 결론 낼 수 있음)"""
        inc = self.store.included()
        used = {i for v in self.syntheses.values() for i in re.findall(r"S\d+", json.dumps(v))}
        must = [s for s in inc if s.sid in used]
        rest = [s for s in inc if s.sid not in used][:max(0, MAX_REPORT_EVIDENCE - len(must))]
        return sorted(must + rest, key=lambda s: -s.score), len(must)

    def write_report(self):
        self.report_evidence, n_used = self._report_evidence()
        evidence = "\n".join(f"[{s.sid}] {s.short_cite()} | {s.kind} | {s.study_type} | sample: {s.sample} | {s.direction} | {s.key_finding}"
                             for s in self.report_evidence)
        log("Report", f"보고서 근거 {len(self.report_evidence)}건 전달 (종합에서 다룬 {n_used}건 전부 포함, 채택 {len(self.store.included())}건 중)")
        critic = self.reflections[-1] if self.reflections else {}
        body = self.llm.text(SYS_BASE + " You are now the LEAD RESEARCHER writing the final research brief.", f"""Write the final research brief in Markdown, in language "{self.lang}" (headings too).
Cite with source ids like [S3] ONLY from the evidence list. Do not add a reference list (it is appended automatically).
Rules:
- Put each citation inside the sentence it supports, before the final period: "... increased [S3]." (not "... increased. [S3]").
- Before claiming that evidence is absent or that only N studies exist, check the FULL evidence list, and phrase it as
  "in the evidence collected for this brief".
- {ABSTRACT_RULE}
- Several versions/copies of the same study (preprint, publisher page, repository) count as ONE study.

Topic: {self.topic}
Topic analysis: {json.dumps(self.understanding, ensure_ascii=False)}
Sub-question syntheses: {json.dumps(self.syntheses, ensure_ascii=False)}
Critic's last review: {json.dumps(critic, ensure_ascii=False)}
Evidence statistics: {json.dumps(self._evidence_stats(), ensure_ascii=False)}
Evidence list ({len(self.report_evidence)} sources):
{evidence}

Required sections (use ## headings, numbered):
1. Executive summary — 5 bullet points a busy researcher must know.
2. Topic understanding — key concepts, operational definitions, variables, scope (including the search window).
3. Key findings by sub-question — with evidence strength labels.
4. Comparison & synthesis — where studies agree/conflict and why (design, sample, context, measurement differences).
5. Research gaps — concrete and specific (population, method, measurement, context, time horizon), each tied to evidence.
6. Future research directions — 3 proposed studies, each with research question, hypotheses, design, sample,
   variables & measurement instruments, analysis method, expected contribution.
7. Must-read papers — top 5 sources to read first and why.
8. Limitations of this brief — evidence quality, coverage, abstract-only analysis.""", effort="high", max_tokens=32000, stream=True)
        log("Report", f"최종 보고서 작성 완료 ({len(body)}자)")
        return body

    # ── ⑧ 인용 검증 → 수정 → 재검증 ──
    @staticmethod
    def _claim_spans(report):
        """보고서를 '주장 + 인용' 단위의 (시작, 끝) 위치로 나눔. 표는 행 전체, 본문은 문장 단위이고,
        인용 번호만 남은 조각(예: '... 증가했다. [S3].')은 앞 문장에 붙여 주장 없이 번호만 검사하는 일을 막음."""
        spans, pos = [], 0
        for line in report.split("\n"):
            start, end = pos, pos + len(line)
            pos = end + 1
            if not _CITE.search(line):
                continue
            if line.lstrip().startswith("|"):
                spans.append((start, end))
                continue
            segs, a = [], start
            for m in _SENT_END.finditer(line):
                segs.append((a, start + m.start()))
                a = start + m.end()
            segs.append((a, end))
            merged = []
            for a, b in segs:
                core = re.sub(r"[\W_]", "", _CITE.sub("", report[a:b]))
                if merged and len(core) < 2:
                    merged[-1] = (merged[-1][0], b)
                else:
                    merged.append((a, b))
            spans += [(a, b) for a, b in merged if _CITE.search(report[a:b])]
        return spans

    def _sources_text(self, claim):
        out = []
        for i in dict.fromkeys(re.findall(r"S\d+", " ".join(_CITE.findall(claim)))):
            s = self.store.items.get(i)
            if s and len(out) < 5:
                out.append(f"  [{s.sid}] {s.title} ({s.year or 'n.d.'}, {s.kind})\n  ABSTRACT: {(s.abstract or 'NO ABSTRACT')[:3000]}")
        return "\n".join(out)

    def _check(self, claims):
        """claims=[주장 문장] → [(판정, 메모)]. 인용된 문헌의 초록 전문과 대조."""
        blocks = [f"CLAIM {n}: {c}\nSOURCES:\n{self._sources_text(c)}" for n, c in enumerate(claims)]
        schema = O(items=A(O(idx=I, verdict=E(*VERDICTS), note=S)))
        res = self.llm.json(SYS_BASE + " You are a strict FACT-CHECKER.", f"""For each claim, judge whether the cited sources' text supports it.
- supported / partially supported / not in source text (the abstract does not say it) / contradicted (the abstract says otherwise).
- Judge only what the claim attributes to the cited sources. Statements about how many studies exist in this brief's
  evidence base (e.g. "the only RCT") and the author's own definitions or table labels are not judged against the source.
- In a table row, judge only the cells that carry citations.
Write note in {self.lang} (one short sentence).

""" + "\n\n".join(blocks), schema, effort="low")
        got = {it["idx"]: (it["verdict"], it["note"]) for it in (res or {}).get("items", []) if 0 <= it["idx"] < len(claims)}
        return [got.get(n) for n in range(len(claims))]

    def _repair(self, items):
        """근거가 부족한 문장을 인용 문헌이 실제로 말하는 범위로 고쳐 씀 → {번호: 수정문}."""
        blocks = [f"CLAIM {n}: {c['claim']}\nVERDICT: {c['verdict']} — {c['note']}\nSOURCES:\n{self._sources_text(c['claim'])}"
                  for n, c in enumerate(items)]
        res = self.llm.json(SYS_BASE + " You are the EDITOR fixing unsupported statements.", f"""Rewrite each claim so that it states only what its cited sources' text supports.
- Keep the citation ids and the Markdown form: a table row stays one table row with the same number of cells,
  a list item stays a list item, and nothing spans multiple lines.
- {ABSTRACT_RULE}
- If a point cannot be supported, soften it (e.g. "suggests", "in one study") or drop that part; do not add new facts.
Write in language "{self.lang}".

""" + "\n\n".join(blocks), O(items=A(O(idx=I, revised=S))), effort="medium")
        return {it["idx"]: it["revised"].strip() for it in (res or {}).get("items", []) if 0 <= it["idx"] < len(items)}

    def _valid_revision(self, old, new):
        if not new or "\n" in new or not _CITE.search(new):
            return False
        if any(i not in self.store.items for i in re.findall(r"S\d+", " ".join(_CITE.findall(new)))):
            return False
        if old.lstrip().startswith("|"):
            return new.startswith("|") and new.count("|") == old.strip().count("|")
        return True

    def verify(self, report):
        ids = set(re.findall(r"S\d+", " ".join(_CITE.findall(report))))
        unknown = sorted(i for i in ids if i not in self.store.items)
        for u in unknown:  # 존재하지 않는 인용 제거
            report = re.sub(rf"\b{u}\b", "?", report)
        spans = self._claim_spans(report)
        picked = spans[:: max(1, len(spans) // VERIFY_SAMPLE)][:VERIFY_SAMPLE] if VERIFY_SAMPLE > 0 else []
        result = {"cited_ids": len(ids), "unknown_ids": unknown, "claims_total": len(spans), "checks": []}
        if not picked:
            log("Verify", f"인용 ID {len(ids)}개, 미존재 {len(unknown)}개, 검증할 인용 문장 없음")
            return report, result
        claims = [report[a:b].strip() for a, b in picked]
        for (a, b), c, v in zip(picked, claims, self._check(claims)):
            if v:
                result["checks"].append({"span": (a, b), "claim": c, "verdict": v[0], "note": v[1]})
        ok = sum(c["verdict"] == "supported" for c in result["checks"])
        log("Verify", f"인용 ID {len(ids)}개, 미존재 {len(unknown)}개, 표본 {len(result['checks'])}문장 중 완전 지지 {ok}")

        # 수정(Self-correction): 완전 지지가 아닌 문장만 근거 범위로 고쳐 쓰고 다시 검증
        bad = [c for c in result["checks"] if c["verdict"] != "supported"]
        if bad:
            revised = self._repair(bad)
            fixes = [(c, revised[n]) for n, c in enumerate(bad) if self._valid_revision(c["claim"], revised.get(n))]
            for (c, new), v in zip(fixes, self._check([new for _, new in fixes]) if fixes else []):
                c["revised"] = new
                c["verdict_after"], c["note_after"] = v if v else ("not checked", "")
            for c, new in sorted(fixes, key=lambda x: -x[0]["span"][0]):  # 뒤에서부터 바꿔야 앞의 위치가 유지됨
                a, b = c["span"]
                lead = report[a:b][:len(report[a:b]) - len(report[a:b].lstrip())]
                report = report[:a] + lead + new + report[b:]
            ok2 = sum((c.get("verdict_after") or c["verdict"]) == "supported" for c in result["checks"])
            log("Repair", f"미지지·부분지지 {len(bad)}문장 중 {len(fixes)}문장 수정 → 재검증 후 완전 지지 {ok2}/{len(result['checks'])}")
        for c in result["checks"]:
            c.pop("span", None)
        return report, result

    # ── 실행 ──
    def run(self):
        t0 = time.time()
        log("Start", f"연구 주제: {self.topic}")
        self.understand()
        self.make_plan()
        self.human_review()
        for sq in self.plan["sub_questions"]:
            self.investigate(sq)
        self.evaluate()
        for sq in self.plan["sub_questions"]:
            self.synthesize(sq)
        for rnd in range(1, MAX_REFLECTION_ROUNDS + 1):
            if not self.reflect(rnd):
                break
            self.replan_and_act(self.reflections[-1])
        else:
            self.reflect(MAX_REFLECTION_ROUNDS + 1, final=True)  # 마지막 추가 조사 이후 상태를 한 번 더 평가
        report, verification = self.verify(self.write_report())
        elapsed = time.time() - t0
        return {"report": report, "verification": verification, "elapsed": elapsed}

In [ ]:
# ⑥ 결과 조립·저장 유틸
LABELS = {
    "ko": dict(title="연구 조사 보고서", topic="연구 주제", gen="생성", model="모델", stats="수집/채택 문헌",
               window="검색 기간", nolimit="제한 없음", since="년 이후", merged="중복 판본 병합", sent="보고서 전달 근거",
               tableA="부록 A. 근거 비교표", refB="부록 B. 참고문헌", verC="부록 C. 인용 검증", traceD="부록 D. 에이전트 수행 기록",
               after="수정 후", before="수정 전", cols=["ID", "문헌 · 유형", "방법 · 표본", "핵심 결과", "방향 · 점수"]),
    "en": dict(title="Research Brief", topic="Topic", gen="Generated", model="Model", stats="Collected/included sources",
               window="Search window", nolimit="no limit", since=" onward", merged="duplicate versions merged",
               sent="sources given to report", tableA="Appendix A. Evidence table", refB="Appendix B. References",
               verC="Appendix C. Citation check", traceD="Appendix D. Agent trace", after="after revision", before="before",
               cols=["ID", "Source · type", "Method · sample", "Key finding", "Direction · score"]),
}
VERDICT_MARK = {"supported": "✅", "partially supported": "🟡", "not in source text": "⚪", "contradicted": "❌"}


def _cell(x, n=160):
    return str(x or "").replace("|", "/").replace("\n", " ")[:n]


def build_markdown(agent, result):
    L = LABELS.get(agent.lang, LABELS["en"])
    inc = agent.store.included()
    window = f"{agent.year_from}{L['since']}" if agent.year_from else L["nolimit"]
    md = [f"# {L['title']}",
          f"**{L['topic']}**: {agent.topic}  ",
          f"{L['gen']}: {datetime.now():%Y-%m-%d %H:%M} · {L['model']}: {agent.llm.model} · {L['window']}: {window} · "
          f"{L['stats']}: {len(agent.store)}/{len(inc)} ({L['merged']} {agent.store.merged}, {L['sent']} {len(agent.report_evidence)}) · "
          f"{result['elapsed'] / 60:.1f} min · "
          f"LLM calls {agent.llm.usage['calls']} (in {agent.llm.usage['input_tokens']:,} / out {agent.llm.usage['output_tokens']:,} tokens)",
          "", "---", "", result["report"], "", "---", f"## {L['tableA']}", "",
          "| " + " | ".join(L["cols"]) + " |", "|" + "---|" * len(L["cols"])]
    for s in inc:
        md.append(f"| {s.sid} | {_cell(s.short_cite())}<br>{_cell(s.study_type)} | {_cell(s.method, 80)}<br>{_cell(s.sample, 60)} "
                  f"| {_cell(s.key_finding, 220)} | {s.direction}<br>{s.score:.2f} |")
    md += ["", f"## {L['refB']}", ""]
    for s in sorted(inc, key=lambda s: int(s.sid[1:])):
        authors = ", ".join(s.authors[:3]) + (" et al." if len(s.authors) > 3 else "")
        md.append(f"- **[{s.sid}]** {authors or s.venue} ({s.year or 'n.d.'}). *{s.title}*. {s.venue}. "
                  f"{s.url} — {s.kind}, cites={s.citations if s.citations is not None else '-'}, via {s.origin}")
    v = result["verification"]
    checks = v["checks"]
    tally = lambda key: " · ".join(f"{VERDICT_MARK[k]} {k} {sum((c.get(key) or c['verdict']) == k for c in checks)}" for k in VERDICTS)
    md += ["", f"## {L['verC']}", "",
           f"- cited ids: {v['cited_ids']} · unknown ids removed: {v['unknown_ids'] or 'none'} · "
           f"claims with citations: {v.get('claims_total', '-')} · checked: {len(checks)}"]
    if checks:
        md += [f"- {L['before']}: {tally('verdict')}", f"- {L['after']}: {tally('verdict_after')}", ""]
    for c in checks:
        md.append(f"- {VERDICT_MARK.get(c['verdict'], '❌')} {c['verdict']} — “{_cell(c['claim'], 150)}…” — {c['note']}")
        if c.get("revised"):
            md.append(f"  - → {L['after']}: {VERDICT_MARK.get(c.get('verdict_after'), '❔')} {c.get('verdict_after')} — "
                      f"“{_cell(c['revised'], 150)}…” — {c.get('note_after', '')}")
    md += ["", f"## {L['traceD']}", ""]
    for t in TRACE:
        if t["stage"] in ("Understand", "Plan", "Act", "Evaluate", "Reflect", "Replan", "Human", "Report", "Verify", "Repair"):
            md.append(f"- `{t['time'][11:]}` **{t['stage']}** — {_cell(t['msg'], 250)}")
        elif t["stage"] == "Tool" and "제외" in t["msg"]:
            md.append(f"- `{t['time'][11:]}` **Tool** — {_cell(t['msg'], 250)}")
    return "\n".join(md)


def to_html(markdown_text, title):
    """인쇄·PDF용 HTML: 긴 표도 칸 안에서 줄바꿈되어 오른쪽이 잘리지 않음 (브라우저에서 열고 인쇄 → PDF로 저장)."""
    try:
        from markdown_it import MarkdownIt
    except ImportError:
        return None
    body = MarkdownIt("commonmark", {"html": True}).enable("table").render(markdown_text)
    css = ("body{font-family:'Noto Sans KR','Malgun Gothic',sans-serif;max-width:1100px;margin:auto;padding:24px;line-height:1.6;font-size:14px}"
           "table{border-collapse:collapse;width:100%;font-size:12px;margin:12px 0}"
           "th,td{border:1px solid #bbb;padding:4px 6px;vertical-align:top;overflow-wrap:break-word;word-break:keep-all}"
           "th{background:#f2f2f2}code{font-size:12px}"
           "@media print{@page{size:A4 landscape;margin:10mm}body{max-width:none;padding:0}tr{break-inside:avoid}}")
    return f"<!doctype html><html lang='ko'><head><meta charset='utf-8'><title>{title}</title><style>{css}</style></head><body>{body}</body></html>"


def save_outputs(agent, markdown):
    import pandas as pd
    slug = re.sub(r"[^\w가-힣]+", "_", agent.topic).strip("_")[:30]
    run_dir = f"{OUTPUT_DIR}/{datetime.now():%Y%m%d_%H%M}_{slug}"   # 실행마다 새 폴더 → 다른 주제 결과와 섞이지 않음
    os.makedirs(run_dir, exist_ok=True)
    with open(f"{run_dir}/research_report.md", "w", encoding="utf-8") as f:
        f.write(markdown)
    html = to_html(markdown, agent.topic)
    if html:
        with open(f"{run_dir}/research_report.html", "w", encoding="utf-8") as f:
            f.write(html)
    rows = [{k: (sorted(v) if isinstance(v, set) else v) for k, v in asdict(s).items()} for s in agent.store.items.values()]
    df = pd.DataFrame(rows, columns=[f.name for f in fields(Source)])   # 수집 0건이어도 열 이름은 유지
    df.sort_values("score", ascending=False).to_csv(f"{run_dir}/evidence_table.csv", index=False, encoding="utf-8-sig")
    with open(f"{run_dir}/agent_trace.json", "w", encoding="utf-8") as f:
        json.dump({"topic": agent.topic, "search_window_from": agent.year_from, "understanding": agent.understanding,
                   "plan": agent.plan, "syntheses": agent.syntheses, "reflections": agent.reflections,
                   "report_evidence": [s.sid for s in agent.report_evidence], "trace": TRACE},
                  f, ensure_ascii=False, indent=2, default=str)
    shutil.make_archive(run_dir, "zip", run_dir)
    print(f"\n💾 저장 완료: {run_dir}/ (research_report.md·.html, evidence_table.csv, agent_trace.json)\n"
          f"   한 번에 내려받기: {run_dir}.zip (Colab 왼쪽 📁 파일 탭에서 우클릭 → 다운로드)")
    return run_dir

In [ ]:
# ⑦ 실행 및 조사 결과 출력
if not RESEARCH_TOPIC.strip():
    raise ValueError("맨 위 셀의 RESEARCH_TOPIC에 연구 주제를 입력하세요.")

TRACE.clear()
agent = ResearchAgent(RESEARCH_TOPIC, LLM(resolve_api_key(), MODEL))
result = agent.run()
final_markdown = build_markdown(agent, result)
run_dir = save_outputs(agent, final_markdown)

try:
    from IPython.display import Markdown, display
    display(Markdown(final_markdown))
except ImportError:
    print(final_markdown)